This is the **set-up cell** - run it first.

* Qiskit has changed a lot over the years, and some older commands (`Aer`, `BasicAer`, `execute`,
  `IBMQ`) no longer exist. The helper file `qiskit_backend_config.py` (in this folder) provides
  stand-ins for them, so the older code below still works.
* `sys.path.insert(...)` tells Python to look for that helper file in the current folder.
* `BACKEND_NAME = "qasm_simulator"` chooses **where the circuits run**. A *backend* is either a
  simulator on your own computer or a real quantum computer at IBM. The `qasm_simulator` imitates
  a real device: it *measures* the qubits many times ("shots") and counts the results.
* If you have an IBM Quantum account you could set a token and pick a real device, but nothing in
  this notebook needs that.
* The last line prints which backend was created, so you know the cell worked.

In [1]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')

Backend ready: LegacyAerSimulator('aer_simulator')


This cell **imports the tools** and **creates two simulators**.

* `numpy` for maths; Qiskit classes to build circuits; `plot_histogram` to draw bar charts of results.
* `state_fidelity` and friends (imported but not used below) measure how similar two quantum
  states are, which is exactly what the swap test estimates *experimentally*.
* `q_simulator` (`qasm_simulator`) behaves like a real device: it measures and counts.
* `s_simulator` (`statevector_simulator`) would give the exact quantum state instead (not used below).

In [2]:
#Amitava Chakraborty - 16th Feb 2020
# Swap Test - Given two unknown quantum states, determine how much them differ

# Importing standard Qiskit libraries and configuring account
import numpy as np
from qiskit import ClassicalRegister, QuantumRegister, QuantumCircuit
from qiskit_backend_config import execute
from qiskit_backend_config import IBMQ, BasicAer
from qiskit.quantum_info import Pauli, state_fidelity, process_fidelity
from qiskit.visualization import plot_histogram

q_simulator = BasicAer.get_backend('qasm_simulator')
s_simulator = BasicAer.get_backend('statevector_simulator')

C:\Users\AMITAVA\AppData\Local\Temp\ipykernel_46268\103654244.py:12: DeprecationWarning: BasicAer is removed in Qiskit 2.x.  Using AerSimulator as a drop-in replacement.
  q_simulator = BasicAer.get_backend('qasm_simulator')
C:\Users\AMITAVA\AppData\Local\Temp\ipykernel_46268\103654244.py:13: DeprecationWarning: BasicAer is removed in Qiskit 2.x.  Using AerSimulator as a drop-in replacement.
  s_simulator = BasicAer.get_backend('statevector_simulator')


This cell **builds a first, simple version of the swap test**.

**The idea.** We want to know whether two quantum states are the same, but we cannot simply look
at a quantum state. The **swap test** uses one extra *ancilla* (helper) qubit:

1. put the ancilla into superposition with a Hadamard ($H$);
2. *controlled-swap* (`cswap`): if the ancilla is 1, swap the two states;
3. apply $H$ to the ancilla again and measure it.

If the two states are **equal**, the ancilla is measured as 0 with probability **100 %**; if they
are **orthogonal** (completely different), with probability **50 %**. In general
$P(0) = \tfrac12 + \tfrac12|\langle\psi|\phi\rangle|^2$.

**In this cell:** qubit 0 stays $|0\rangle$, `x(qr[1])` makes qubit 1 $|1\rangle$ (two orthogonal
states), and qubit 2 is the ancilla. `barrier()` draws a visual separator and does not change the state.

> **Look closely:** the second Hadamard (step 3) is *commented out* (`#circuit.h(qr[2])`). Without
> it, the ancilla always gives 0 or 1 with 50 % each, whatever the two states are, so this version
> cannot tell equal states from different ones. Remove the `#` to get the real swap test.

In [3]:
qr = QuantumRegister(3)  # Initialize qubits
cr = ClassicalRegister(1)  # Initialize bits for record measurements
circuit = QuantumCircuit(qr, cr)

# Create 'unknown' states
circuit.x(qr[1])
circuit.barrier()

# Swap test
circuit.h(qr[2])
circuit.cswap(qr[2], qr[0], qr[1])
#circuit.h(qr[2])

circuit.barrier()

# Measure
circuit.measure(qr[2], cr[0])
# If unknown states are: 
# 1. Orthogonal, then 0 is measured with probability 50%
# 2. Equal, then 0 is measured with probability 100%

This cell **runs the circuit 1024 times** on the simulator and **prints the counts** (how often
the ancilla gave 0 and 1).

* `execute(...)` sends the circuit to the backend; `.result().get_counts()` collects the results.
* The output is about 50 % / 50 %. Here this is expected for two reasons: the states $|0\rangle$
  and $|1\rangle$ are orthogonal, *and* the second Hadamard is missing (see the note above).

In [4]:
# Run the circuit with local simulator
results = execute(circuit, backend=q_simulator, shots=1024).result()
answer = results.get_counts()
print(answer)
# Since our states are orthogonal (qr[0] = |0>, qr[1] = |1>)
# 0 is observed with probability approx. 50%  and
# 1 is observed with probability approx. 50%.

{'0': 511, '1': 513}


Say, $\left\vert \psi \right\rangle$ can be either of two values, a or b. Can we check the value of $\left\vert \psi \right\rangle$?

We may check for state equality with the SWAP test.
[Quantum fingerprinting (Buhrman, Cleve, Watrous & de Wolf, 2001)](https://arxiv.org/abs/quant-ph/0102001) seems to be the first paper to introduce the SWAP test.

The idea behind this test is:

 1. Encode the 2 quantum states using quantum error correction codes to "increase the difference between them".
 2. Test the 2 code words by using an ancilla register and the procedure below.
 3. Read the ancilla register. If it is $\left\vert 0 \right\rangle$ then the 2 states are probably equal. Else, they are probably different. 

<img src='images/swapTest.png' alt="" width="500" align="middle"/>

We can repeat the procedure multiple times to ensure that the 2 states are equal up to a given probability.

This cell runs **a complete swap test on two 2-qubit states**, $|\psi\rangle$ and $|\phi\rangle$.

**The registers.** `qr_psi` (2 qubits) holds $|\psi\rangle$, `qr_phi` (2 qubits) holds
$|\phi\rangle$, `qr_ancilla` (1 qubit) is the helper, and `cequal` (1 classical bit) stores the
answer.

**Two helper functions** (`def` defines a reusable function):

* `cswap` swaps two registers **qubit by qubit**, but only if the control qubit is 1. Each swap is
  built from three Toffoli (`ccx`) gates, a classic trick: three controlled-XORs make a swap.
  The `assert` line stops the program with an error message if the registers have different sizes.
* `equality_test` is the swap test: $H$ on the ancilla → controlled swap → $H$ → measure.

**Preparing the states.**

* $|\psi\rangle$: `h(qr_psi)` gives $|{+}{+}\rangle$, an equal superposition on both qubits.
* $|\phi\rangle$: `x`, then `y`, then `h`. Step by step on each qubit:
  $|0\rangle \xrightarrow{X} |1\rangle \xrightarrow{Y} -i|0\rangle \xrightarrow{H} -i|{+}\rangle$.
  So $|\phi\rangle = (-i)^2|{+}{+}\rangle = -|{+}{+}\rangle$. That is the **same state** as $|\psi\rangle$,
  apart from a factor $-1$ (a *global phase*, which no measurement can detect).

**Result.** Because the states are equal, the output is `{'0': 1024}`: the ancilla gave 0 every
single time.

**Try it:** change the preparation of `qr_phi` (e.g. delete the `circuit.h(qr_phi)` line) and see
how the counts change.

In [5]:
register_size = 2

qr_psi = QuantumRegister(register_size, 'psi')  #For state PSI
qr_phi = QuantumRegister(register_size, 'phi')  #For state PHI
qr_ancilla = QuantumRegister(1, 'ancilla')
cequal = ClassicalRegister(1, 'equal')

circuit = QuantumCircuit()

circuit.add_register(qr_psi)
circuit.add_register(qr_phi)
circuit.add_register(qr_ancilla)
circuit.add_register(cequal)


def cswap(circuit, ctrl, q1, q2) -> None:
    assert len(q1) == len(q2), "The swapped register sizes should match"
    for i in range(len(q1)):
        # Controlled swap
        circuit.ccx(ctrl, q1[i], q2[i])
        circuit.ccx(ctrl, q2[i], q1[i])
        circuit.ccx(ctrl, q1[i], q2[i])

def equality_test(circuit, ancilla, q1, q2, classical_register) -> None:
    assert len(q1) == len(q2), "The swapped register sizes should match"
    circuit.h(ancilla[0])
    cswap(circuit, ancilla[0], q1, q2)
    circuit.h(ancilla[0])
    circuit.measure(ancilla[0], classical_register[0])

## Initialisation
# We add Hadamard to all the registers - to create PSI
circuit.x(qr_phi)
circuit.h(qr_psi)
# We add Hadamard to all the registers - to create PHI
# Modify the initialisation of either PHI or PSI and check the results.
circuit.y(qr_phi)
circuit.h(qr_phi)

## SWAP test
equality_test(circuit, qr_ancilla, qr_psi, qr_phi, cequal)

res_qasm = execute([circuit], q_simulator, shots=1024).result()
counts = res_qasm.get_counts()

print(counts)

{'0': 1024}




[1]: images/swapTest.png